# Get to Know a Dataset: THEIA-Pancreas

This notebook is a guided tour of the [THEIA-Pancreas](https://registry.opendata.aws/theia-pancreas) dataset: human pancreas whole-slide images (WSIs) from 284 organ donors, 223 expert-annotated pancreatic ductal adenocarcinoma (PDAC) slides, and the DHCP semantic segmentation benchmark. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

**This is an onboarding draft.** The S3 bucket is not populated yet. Until it is, this notebook reads a local copy of the data, using paths taken from environment variables. Each step notes what the S3 version may look like.

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

**This is a preliminary draft of what the bucket structure may end up looking like.**

THEIA-Pancreas has three components, each under its own prefix in `s3://theia-pancreas/`:

1. **Donor pancreas atlas**: 1,687 WSIs from 284 organ donors, sampled across the pancreatic head, body, tail, and uncinate process (`data/wsi/donor-atlas/`). Donors may have non-cancer pathologies or be completely normal. Annotations for PanINs and IPMNs are currently being created.
2. **Annotated PDAC slides**: 223 WSIs from a neoadjuvant-treated UT Southwestern (UTSW) cohort (50 slides) and the treatment-naive CPTAC-PDA cohort (173 slides) (`data/wsi/pdac/`), with more than 140,000 hand-drawn polygon annotations (`data/annotations/`).
3. **DHCP benchmark**: 75,519 pre-tiled 1024x1024 image/mask pairs derived from the annotated slides, split 70/20/10 at the patient level (`data/benchmark/dhcp/`).

```
s3://theia-pancreas/
  README.txt
  LICENSE.txt
  metadata/
    slide_manifest.csv                      # donor atlas slide index
    dhcp-croissant.json                     # MLCommons Croissant metadata for DHCP
  data/
    wsi/
      donor-atlas/donor_id=<DONOR_ID>/*.svs
      pdac/cohort=utsw/*.{svs,ndpi,qptiff}
      pdac/cohort=cptac-pda/*.jpg
    annotations/json/
      cohort=utsw/*.json
      cohort=cptac-pda/*.json
    benchmark/dhcp/v1.0.0/
      classes_to_int.tsv
      {train,test,validation}_data/{images,masks}/
```

Every file is a separate object (no archives), so a single slide, annotation file, or tile can be fetched on its own. The partition-style prefixes (`donor_id=`, `cohort=`) make the layout self-describing, and the benchmark is versioned so published results stay reproducible.

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, uv, conda):
#
# openslide-python >= 1.4
# openslide-bin >= 4.0
# boto3 >= 1.38
# numpy >= 2.0
# pandas >= 2.2
# matplotlib >= 3.10
# pillow >= 11.0

# Built-ins
import os
import random
import xml.etree.ElementTree as ET
from pathlib import Path

# Installed libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Polygon
from PIL import Image
import openslide

#### Setup: data locations

Until the bucket is live, point these environment variables at a local copy of the data:

| Variable | Points to |
|---|---|
| `THEIA_MANIFEST` | `metadata/slide_manifest.csv` |
| `THEIA_WSI_DIR` | a folder of UTSW WSIs (e.g. `data/wsi/pdac/cohort=utsw/`) |
| `THEIA_ANNOTATIONS_DIR` | `data/annotations/json/` (containing `utsw/` and `cptac-pda/`) |
| `THEIA_DHCP_DIR` | `data/benchmark/dhcp/v1.0.0/` |

Once the bucket is live, listing it will look like this (no AWS account needed):

```python
import boto3
from botocore import UNSIGNED
from botocore.config import Config

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
for item in s3.list_objects_v2(Bucket="theia-pancreas", Delimiter="/")["CommonPrefixes"]:
    print(item["Prefix"])
```

In [ ]:
MANIFEST = Path(os.environ["THEIA_MANIFEST"])
WSI_DIR = Path(os.environ["THEIA_WSI_DIR"])
ANNOTATIONS_DIR = Path(os.environ["THEIA_ANNOTATIONS_DIR"])
DHCP_DIR = Path(os.environ["THEIA_DHCP_DIR"])

# Shared colors for the 8 DHCP tissue classes, used in every figure below
CLASS_COLORS = {
    "tumor": "#2a78d6",
    "artery": "#eb6834",
    "vessel": "#1baf7a",
    "nerve": "#eda100",
    "lymphoid_aggregate": "#e87ba4",
    "normal_pancreas": "#008300",
    "normal_epithelium": "#6250d6",
    "adipose": "#e34948",
}
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#8a8a85", "axes.labelcolor": "#3d3d3a",
    "xtick.color": "#5c5c58", "ytick.color": "#5c5c58",
    "axes.titleweight": "bold", "figure.dpi": 110,
})

The DHCP benchmark is the most deeply nested component. Each split holds an `images/` and a `masks/` prefix, and each tile has the same file stem in both.

In [ ]:
for split in sorted(p.name for p in DHCP_DIR.iterdir() if p.is_dir()):
    for kind in ("images", "masks"):
        files = sorted((DHCP_DIR / split / kind).iterdir())
        print(f"{split}/{kind}/  {len(files):>6,} files   e.g. {files[0].name}")
print((DHCP_DIR / "classes_to_int.tsv").read_text())

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

**Whole-slide images.** A WSI is a digitized glass slide, typically scanned at 40x magnification (about 0.22-0.25 microns per pixel), often over 100,000 pixels on a side. To make them practical to view and analyze, scanners store them as *pyramidal, tiled TIFFs*: the full-resolution image plus progressively downsampled copies, each cut into small tiles. A reader only decodes the tiles it needs at the resolution it needs, so a thumbnail or a small region loads in milliseconds from a multi-gigabyte file. THEIA-Pancreas contains the following formats, which may or may not correspond to original formats out of the scanner:

- **SVS** (`.svs`): donor atlas and most UTSW slides
- **NDPI** (`.ndpi`): UTSW slides
- **QPTIFF** (`.qptiff`): one UTSW slide
- **JPEG** (`.jpg`): CPTAC-PDA regions, cropped from the original [TCIA](https://doi.org/10.7937/k9/tcia.2018.sc20fo18) slides before annotation

[OpenSlide](https://openslide.org/) reads all of the pyramidal formats through one API. `pip install openslide-python openslide-bin` installs it with no system libraries needed. OpenSlide reads local files, so download a slide first (`aws s3 cp --no-sign-request ...`). On Amazon EC2 or Amazon SageMaker in the same region as the bucket, a slide downloads in seconds.

**Annotations.** Polygon annotations will be provided in a documented JSON format, with one file per slide. Each element will correspond to a single structure with a tissue `label` and a `points` list of `x,y` vertices in source image pixel space. Polygons convert easily to `shapely` geometries or rasterized masks.

**DHCP benchmark.** Tiles are standard RGB **JPEG** files. Masks are single-channel 8-bit **PNG** files where each pixel value is a class integer (`classes_to_int.tsv`). These load with any image library and need no special tooling, which makes the benchmark easy to use with PyTorch, TensorFlow, or Amazon SageMaker training jobs. Benchmark metadata is also provided in [MLCommons Croissant](https://mlcommons.org/croissant/) format (`metadata/dhcp-croissant.json`).

**Tabular metadata.** `slide_manifest.csv` indexes the donor atlas: one row per slide, with `donor_id`, `region` (`H` head, `B` body, `T` tail, `U` uncinate, `A` other, `SPLEEN` spleen), and `replicate`.

### Q: Can you show us an example of downloading and loading data from your dataset?

We start with the donor atlas manifest, which is the index to the largest part of the dataset.

In [ ]:
manifest = pd.read_csv(MANIFEST, usecols=["filename", "donor_id", "region", "replicate"])
print(f"{len(manifest):,} slides from {manifest.donor_id.nunique()} donors")
manifest.head()

Next, open one of the annotated UTSW slides with OpenSlide. A single file holds several resolution levels.

In [ ]:
slide_path = sorted(p for p in WSI_DIR.iterdir() if p.suffix in {".svs", ".ndpi", ".qptiff"})[0]
slide = openslide.OpenSlide(slide_path)

print(slide_path.name, "| vendor:", slide.properties.get("openslide.vendor"),
      "| objective:", slide.properties.get("openslide.objective-power") + "x",
      "| microns per pixel:", round(float(slide.properties["openslide.mpp-x"]), 3))
pd.DataFrame({
    "width": [w for w, h in slide.level_dimensions],
    "height": [h for w, h in slide.level_dimensions],
    "downsample": slide.level_downsamples,
}).rename_axis("level")

The slide's annotations are currently in the matching CVAT XML file, **but will be converted to JSON prior to release**. We parse every polygon into a table row, keeping its label, vertices, and any attributes.

In [ ]:
def load_cvat_polygons(xml_path):
    """Read a CVAT for images 1.1 XML file into a DataFrame, one row per polygon."""
    rows = []
    for image in ET.parse(xml_path).getroot().iter("image"):
        for poly in image.iter("polygon"):
            pts = np.array([p.split(",") for p in poly.get("points").split(";")], dtype=float)
            attrs = {a.get("name"): a.text == "True" for a in poly.findall("attribute")}
            rows.append({"slide": image.get("name"), "label": poly.get("label"), "points": pts, **attrs})
    return pd.DataFrame(rows)

polygons = load_cvat_polygons(ANNOTATIONS_DIR / "utsw" / f"{slide_path.stem}.xml")
print(f"{len(polygons):,} polygons on {slide_path.name}")
polygons.label.value_counts().to_frame("polygons").T

Finally, load one benchmark tile and its mask. The file name encodes the source slide and the tile's bounding box in source image slide pixel space: `{slide_id}_{left}_{upper}_{right}_{lower}`.

In [ ]:
classes = pd.read_csv(DHCP_DIR / "classes_to_int.tsv", sep=r"\s+", header=None, names=["class_name", "class_int"])
id_to_class = dict(zip(classes.class_int, classes.class_name))

tile_path = sorted((DHCP_DIR / "train_data" / "images").iterdir())[0]
tile = np.asarray(Image.open(tile_path))
mask = np.asarray(Image.open(DHCP_DIR / "train_data" / "masks" / f"{tile_path.stem}.png"))

*slide_parts, left, upper, right, lower = tile_path.stem.split("_")
print("source slide:", "_".join(slide_parts), "| bbox:", (left, upper, right, lower))
print("tile:", tile.shape, tile.dtype, "| mask:", mask.shape, mask.dtype)
print("classes present:", [id_to_class[v] for v in np.unique(mask)])

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

**Sampling across the pancreas.** Most atlas donors are sampled at the head, body, and tail of the pancreas, with two replicate sections per region. This supports within-donor comparisons along the organ, not only between donors.

**This may change prior to release.** For example, we will likely exclude the spleen and other samples.

In [ ]:
region_names = {"H": "Head", "B": "Body", "T": "Tail", "U": "Uncinate", "A": "Other", "SPLEEN": "Spleen"}
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

counts = manifest.region.map(region_names).value_counts()
ax1.barh(counts.index[::-1], counts.values[::-1], color="#2a78d6", height=0.6)
for y, v in enumerate(counts.values[::-1]):
    ax1.text(v + 8, y, f"{v:,}", va="center", color="#3d3d3a", fontsize=9)
ax1.set_title("Atlas slides by sampling site")
ax1.set_xlabel("Slides")

per_donor = manifest.groupby("donor_id").size().value_counts().sort_index()
ax2.bar(per_donor.index.astype(str), per_donor.values, color="#2a78d6", width=0.6)
for x, v in enumerate(per_donor.values):
    ax2.text(x, v + 4, f"{v:,}", ha="center", color="#3d3d3a", fontsize=9)
ax2.set_title("Donors by number of slides")
ax2.set_xlabel("Slides per donor")
ax2.set_ylabel("Donors")
for ax in (ax1, ax2):
    ax.grid(axis="x" if ax is ax1 else "y", color="#e4e3dd", linewidth=0.8)
    ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

**Expert annotations on whole slides.** Many annotated PDAC slides have thousands of polygon annotations. Below, polygons are drawn over the slide thumbnail, grouped into the eight DHCP tissue classes.

In [ ]:
# Group the raw annotation labels into the 8 DHCP classes for display
LABEL_GROUPS = {
    "tumor": "tumor", "tumor_poorly_differentiated": "tumor", "single_tumor_cell": "tumor",
    "artery": "artery", "vessel": "vessel", "vein": "vessel", "nerve": "nerve",
    "tertiary_lymphoid_structure": "lymphoid_aggregate", "lymph_node": "lymphoid_aggregate",
    "acinar_cells": "normal_pancreas", "beta_islet": "normal_pancreas",
    "ductal_cells": "normal_epithelium", "adipose": "adipose",
}

def draw_polygons(ax, polys, scale=1.0, offset=(0, 0), lw=0.6, fill_alpha=0.25):
    for _, row in polys.iterrows():
        group = LABEL_GROUPS.get(row.label)
        if group is None:
            continue
        pts = (row.points - offset) / scale
        color = CLASS_COLORS[group]
        ax.add_patch(Polygon(pts, closed=True, facecolor=color, alpha=fill_alpha, edgecolor="none"))
        ax.add_patch(Polygon(pts, closed=True, facecolor="none", edgecolor=color, linewidth=lw))

legend_handles = [Patch(facecolor=c, label=n.replace("_", " ")) for n, c in CLASS_COLORS.items()]

wsi_paths = sorted(p for p in WSI_DIR.iterdir() if p.suffix in {".svs", ".ndpi", ".qptiff"})
fig, axes = plt.subplots(len(wsi_paths), 1, figsize=(13, 5.5 * len(wsi_paths)))
for ax, path in zip(np.atleast_1d(axes), wsi_paths):
    wsi = openslide.OpenSlide(path)
    thumb = wsi.get_thumbnail((2400, 2400))
    scale = wsi.dimensions[0] / thumb.size[0]
    ax.imshow(thumb)
    draw_polygons(ax, load_cvat_polygons(ANNOTATIONS_DIR / "utsw" / f"{path.stem}.xml"), scale=scale)
    ax.set_title(f"{path.name}  ({wsi.dimensions[0]:,} x {wsi.dimensions[1]:,} px)")
    ax.axis("off")
fig.legend(handles=legend_handles, loc="lower center", ncol=8, frameon=False, fontsize=10)
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.show()

**Zooming in.** Because WSIs are pyramidal, we can jump straight to full resolution without loading the whole slide. Here we read a 2,048 x 2,048 pixel region (about 0.5 mm across) around a nerve and draw the annotations that fall inside it.

In [ ]:
REGION = 2048
nerves = polygons[polygons.label == "nerve"].copy()
nerves["area"] = nerves.points.apply(lambda p: np.ptp(p[:, 0]) * np.ptp(p[:, 1]))
nerves = nerves[nerves.area < (REGION * 0.6) ** 2]           # a nerve that fits in the window
center = nerves.sort_values("area").iloc[-1].points.mean(axis=0)
origin = (center - REGION / 2).astype(int)

region = slide.read_region(tuple(origin), 0, (REGION, REGION)).convert("RGB")
in_view = polygons[polygons.points.apply(
    lambda p: ((p >= origin) & (p <= origin + REGION)).all(axis=1).any())]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6.8))
for ax in (ax1, ax2):
    ax.imshow(region)
    ax.set_xlim(0, REGION); ax.set_ylim(REGION, 0)
    ax.axis("off")
draw_polygons(ax2, in_view, offset=origin, lw=1.5, fill_alpha=0.15)
ax1.set_title("Full-resolution H&E")
ax2.set_title("With expert annotations")
present = sorted({LABEL_GROUPS[l] for l in in_view.label if l in LABEL_GROUPS}, key=list(CLASS_COLORS).index)
fig.legend(handles=[h for h in legend_handles if h.get_label() in {p.replace("_", " ") for p in present}],
           loc="lower center", ncol=len(present), frameon=False, fontsize=10)
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

**Benchmark tiles.** Each DHCP tile pairs an H&E image with a pixel-level mask. Here are a few random training tiles that contain at least three tissue classes.

In [ ]:
random.seed(8)
train_images = sorted((DHCP_DIR / "train_data" / "images").iterdir())
examples = []
for path in random.sample(train_images, 400):
    m = np.asarray(Image.open(DHCP_DIR / "train_data" / "masks" / f"{path.stem}.png"))
    if len(set(np.unique(m)) - {0}) >= 3:
        examples.append((path, m))
    if len(examples) == 4:
        break

palette = np.array([[255, 255, 255]] + [[int(c[i:i + 2], 16) for i in (1, 3, 5)] for c in CLASS_COLORS.values()], dtype=np.uint8)
fig, axes = plt.subplots(2, len(examples), figsize=(3.3 * len(examples), 6.8))
for col, (path, m) in enumerate(examples):
    img = np.asarray(Image.open(path))
    overlay = np.where(m[..., None] > 0, (0.45 * img + 0.55 * palette[m]).astype(np.uint8), img)
    axes[0, col].imshow(img)
    axes[1, col].imshow(overlay)
    axes[0, col].set_title("_".join(path.stem.split("_")[:-4]), fontsize=9, fontweight="normal")
for row, label in enumerate(("H&E tile", "Mask overlay")):
    axes[row, 0].text(-0.06, 0.5, label, transform=axes[row, 0].transAxes, rotation=90, va="center", ha="right", color="#3d3d3a")
for ax in axes.flat:
    ax.axis("off")
fig.legend(handles=legend_handles, loc="lower center", ncol=4, frameon=False, fontsize=10)
plt.tight_layout(rect=(0.02, 0.08, 1, 1), h_pad=1.5)
plt.show()

**What is in the benchmark?** Counting mask pixels over a random sample of training tiles shows that the two cohorts look very different. Aside from significant institutional differences (the CPTAC-PDA cohort is sourced from multiple centers internationally), the UTSW cohort received neoadjuvant chemotherapy and the CPTAC-PDA cohort is treatment-naive. Models trained on one cohort therefore face a real domain shift on the other, which makes DHCP a useful test of generalization.

In [ ]:
SAMPLE_PER_COHORT = 300
random.seed(0)
cohort_of = lambda p: "CPTAC-PDA" if p.name.startswith("C3") else "UTSW"
by_cohort = {c: [p for p in train_images if cohort_of(p) == c] for c in ("UTSW", "CPTAC-PDA")}

fractions = {}
for cohort, paths in by_cohort.items():
    counts = np.zeros(len(id_to_class), dtype=np.int64)
    for path in random.sample(paths, SAMPLE_PER_COHORT):
        m = np.asarray(Image.open(DHCP_DIR / "train_data" / "masks" / f"{path.stem}.png"))
        counts += np.bincount(m.ravel(), minlength=len(id_to_class))
    fractions[cohort] = counts[1:] / counts[1:].sum()   # share of annotated (non-background) pixels
prevalence = pd.DataFrame(fractions, index=[id_to_class[i] for i in range(1, len(id_to_class))])

fig, ax = plt.subplots(figsize=(11, 4.5))
y = np.arange(len(prevalence))
for offset, cohort, color in ((-0.2, "UTSW", "#2a78d6"), (0.2, "CPTAC-PDA", "#eb6834")):
    ax.barh(y + offset, prevalence[cohort] * 100, height=0.36, color=color, label=f"{cohort} (n={len(by_cohort[cohort]):,} tiles)")
ax.set_yticks(y, [n.replace("_", " ") for n in prevalence.index])
ax.invert_yaxis()
ax.set_xlabel("Share of annotated pixels (%)")
ax.set_title(f"Tissue composition by cohort ({SAMPLE_PER_COHORT} random training tiles each)", pad=30)
ax.grid(axis="x", color="#e4e3dd", linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2)
plt.tight_layout()
plt.show()
(prevalence * 100).round(1)

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

*To be completed. We are finalizing which of our analyses to present here, and will add a worked, reproducible example in a future revision of this notebook.*

Please see our [bioRxiv paper](https://www.biorxiv.org/content/10.1101/2025.08.12.667175v1.abstract) for examples that may be included in the future.

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

**Challenge: Does whole-slide context improve tile-level segmentation?**

Most segmentation models for histopathology, including the baselines we trained on DHCP, see
one tile at a time. A 1024x1024 tile at 0.5 µm per pixel covers about half a millimeter of
tissue. Pathologists, however, rarely label a structure from such a small field of view.
Whether a gland is tumor or normal duct, or whether a vessel is an artery or a vein, often
depends on what surrounds it: the tissue architecture, nearby tumor, and the overall region
of the slide.

THEIA-Pancreas makes this question testable. Alongside the pre-tiled DHCP benchmark, we
release the whole-slide images and full annotations the tiles were cut from. Every tile's
file name records its source slide and bounding box, so each tile can be placed back into its
slide and given any amount of surrounding context. Because the train/test/validation splits
are made at the patient level, every slide belongs to exactly one split, so context taken
from a tile's own slide never crosses into another split.

We would like to know:
- Do multi-scale or context-aware models (for example, a low-magnification view of the
surrounding region alongside each tile, or slide-level embeddings from a pathology foundation
model) beat tile-only models on the DHCP test set?
- Which classes benefit most? We expect context to matter most for classes defined by their
surroundings, such as normal pancreas vs. tumor, artery vs. vessel, and lymphoid aggregates,
and least for classes recognizable from texture alone, such as adipose.
- How much context is enough, and what does it cost in compute?

Advice:
- **Keep the comparison fair.** Report results on the unchanged DHCP test split, with the
same metrics as the tile-only baselines (Dice, IoU, HD95), so gains are attributable to
context alone. For slides in training and validation splits, any approach can be used, including
leveraging WSIs.
- **Use unlabeled context only.** For test tiles, the surrounding image is fair to use, as it
would be in practice, but the surrounding test-set annotations are not.
- **Mind the coordinates.** Tile coordinates refer to the source image: full-resolution slide
pixels for UTSW slides, and pixels of the cropped JPEGs for CPTAC-PDA, whose available
context is limited to the cropped region.
- **Start small.** A two-scale model (each tile plus a downsampled 4x-larger window centered
on it) is a simple first experiment before moving to full-slide encoders.

**Further open questions**

- **Mapping the donor pancreas.** The donor atlas has 1,687 slides from 284 donors, mostly
sampled at the head, body, and tail, but no pixel-level labels. Can models trained on DHCP
measure tissue composition (acinar tissue, islets, ducts, adipose) along the organ and across
donors? Replicate sections from the same donor and region give a built-in check on
consistency.
- **Self-supervised learning beyond cancer.** Pathology foundation models are pretrained
largely on slides from oncology archives, so non-neoplastic tissue such as healthy acinar
parenchyma, islets, and normal ducts may be underrepresented. The donor atlas offers 1,687 slides
of non-cancerous pancreas from 284 donors. How does continued self-supervised
pretraining on these slides change what a model learns? Do adapted representations separate
normal structures (for example islets vs. acinar tissue) more cleanly, capture variation
along the organ (head, body, and tail) or between donors, or improve DHCP segmentation of the
normal-tissue classes? And is anything lost on tumor in the process?

---
*THEIA-Pancreas is released under CC BY 4.0. These data are for research use only and are not suitable for clinical or diagnostic use. Questions and feedback: william.gasper@utsouthwestern.edu or open an issue in this repository.*